[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/01_frozen_mert_classifier_head.ipynb)


# 01 - Classical Baseline and Frozen MERT Classifier Heads

## Research question

Which lightweight classifier is sufficient once MERT representations are frozen, and how much does MERT improve over classical timbral descriptors?

## Approach

 The neural head ablation reuses one MERT-v1-95M last-layer mean-pooling cache, so the only neural factors changed are head type, hidden width, dropout, and one learning-rate setting.

## What is fixed

Unless a selected config says otherwise: MedleyDB instrument labels, the `largest_balanced` split, MERT-v1-95M, 5 s segments, validation-based model selection, and test metrics saved only after training.

## What is varied

Frozen MERT heads: linear, MLP hidden 256 with dropout 0.0/0.2/0.5, MLP hidden 512 dropout 0.2, and MLP hidden 256 dropout 0.2 with learning rate 3e-4.

## Expected interpretation

If the representation is already strong, the linear head should be competitive; larger heads or dropout should help only when validation macro-F1 improves without unstable test behavior.


## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- `EXPERIMENT_GROUPS`, `DATASET_GROUPS`, `MIXTURE_GROUPS`, and `SELECTED_EXPERIMENTS` to run one config at a time.
- `REPLACE_EXISTING = True` only when intentionally overwriting a completed run.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR, RUN_ROOT / "configs"]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["HF_HOME"] = str(RUN_ROOT / "huggingface")

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("METADATA_DIR:", METADATA_DIR)
print("METADATA_DIR exists:", METADATA_DIR.is_dir())

print("CACHE_ROOT:", CACHE_ROOT)
print("CACHE_ROOT exists:", CACHE_ROOT.is_dir())

print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.is_dir())

print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path
import subprocess
import shutil
from google.colab import drive

drive_path = Path("/content/drive")
backup_path = Path("/content/drive_local_backup_before_real_mount")

mount_info = subprocess.run(
    "mount | grep -E '/content/drive|drivefs|google' || true",
    shell=True,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print("Mount info:")
print(mount_info.stdout if mount_info.stdout.strip() else "No Google Drive mount detected.")

# If Drive is not really mounted, move away the local fake folder and mount real Drive
if not mount_info.stdout.strip():
    print("\nNo real Drive mount detected.")

    if drive_path.exists():
        if backup_path.exists():
            shutil.rmtree(backup_path)
        print("Moving existing local /content/drive to:", backup_path)
        shutil.move(str(drive_path), str(backup_path))

    drive_path.mkdir(parents=True, exist_ok=True)
    drive.mount("/content/drive")
else:
    print("\nDrive appears to be mounted already.")

print("\nChecking Drive paths after mount check:")

paths_to_check = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/medleydb_mert_project"),
    Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB/Audio"),
]

for p in paths_to_check:
    print(p, "->", p.exists(), "| is_dir:", p.is_dir())

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
from google.colab import userdata

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

token = userdata.get("GITHUB_TOKEN")
if token is None:
    raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = "https://github.com/Frabbandera/medleydb-mert-instrument-classification.git"

if PROJECT_ROOT.exists():
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
else:
    print("Repository not found. Cloning...")
    subprocess.run(["git", "clone", repo_url, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["HF_HOME"] = str(RUN_ROOT / "huggingface")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("\nSETUP COMPLETE")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("METADATA_DIR:", METADATA_DIR)
print("METADATA_DIR exists:", METADATA_DIR.is_dir())

print("CACHE_ROOT:", CACHE_ROOT)
print("CACHE_ROOT exists:", CACHE_ROOT.is_dir())

print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.is_dir())

print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path

subset_path = METADATA_DIR / "subset_largest_balanced_medleydb_instrument.csv"
label_path = METADATA_DIR / "labels_largest_balanced_medleydb_instrument_label_to_id.json"

mert_cache_dir = (
    CACHE_ROOT
    / "mert_v1_95m"
    / "largest_balanced"
    / "medleydb_instrument"
    / "layer_last_pool_mean"
)

required_files = [
    subset_path,
    label_path,
    mert_cache_dir / "train.pt",
    mert_cache_dir / "val.pt",
    mert_cache_dir / "test.pt",
    mert_cache_dir / "embedding_config.json",
    RESULTS_DIR / "experiment_registry.csv",
]

for path in required_files:
    print(path, "->", path.is_file())

print("\nMERT cache dir:")
print(mert_cache_dir)

In [ ]:
def q(path):
    return shlex.quote(str(path))


def run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)


def load_config(config_path):
    return yaml.safe_load(Path(config_path).read_text(encoding="utf-8"))


def run_experiment_config(config_path, *, extract_embeddings=True, replace_existing=False):
    config_path = Path(config_path)
    cfg = load_config(config_path)
    approach = cfg.get("approach", "frozen_embeddings")
    if approach == "frozen_embeddings" and extract_embeddings:
        run(f"python -m src.features.extract_mert_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    elif approach == "polyphonic_multilabel" and extract_embeddings:
        run(f"python -m src.features.extract_mert_mixture_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    args = f"--config {q(config_path)}"
    if replace_existing:
        args += " --replace-existing"
    return run(f"python -m src.experiments.run_experiment {args}")


def run_selected(configs, *, extract_embeddings=True, replace_existing=False):
    for config in configs:
        run_experiment_config(config, extract_embeddings=extract_embeddings, replace_existing=replace_existing)


## Experiment Groups

Change `SELECTED_EXPERIMENTS` to one or more config paths. The MERT classifier-head configs all reuse `data/cache/mert_v1_95m/largest_balanced/medleydb_instrument/layer_last_pool_mean`; the classical baseline does not use a MERT cache.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

EXPERIMENT_GROUPS = {
    "classical_baseline": [
        "configs/experiments/classical_largest_balanced_medleydb_mfcc_svm.yaml",
    ],
    "mert_head_ablation": [
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_linear.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4.yaml",
    ],
}

all_configs = sum(EXPERIMENT_GROUPS.values(), [])

print("Checking config files:")
for cfg_path in all_configs:
    path = Path(cfg_path)
    print(cfg_path, "->", path.is_file())

print("\nExperiment IDs:")
for cfg_path in all_configs:
    with Path(cfg_path).open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    print(Path(cfg_path).name, "->", cfg["experiment_id"], "| approach:", cfg["approach"])

registry_path = RESULTS_DIR / "experiment_registry.csv"
print("\nRegistry exists:", registry_path.is_file())

if registry_path.is_file():
    registry = pd.read_csv(registry_path)
    existing_ids = set(registry["experiment_id"].astype(str))
    print("\nAlready registered among notebook 01 official configs:")
    for cfg_path in all_configs:
        with Path(cfg_path).open("r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
        print(cfg["experiment_id"], "->", cfg["experiment_id"] in existing_ids)

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")

print("Before:")
print("Current working directory:", Path.cwd())
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("PROJECT_ROOT/configs exists:", (PROJECT_ROOT / "configs").is_dir())

if not PROJECT_ROOT.exists():
    raise RuntimeError("PROJECT_ROOT does not exist. You need to rerun the clone/setup cell first.")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("\nAfter:")
print("Current working directory:", Path.cwd())
print("configs/experiments exists:", Path("configs/experiments").is_dir())
print("requirements.txt exists:", Path("requirements.txt").is_file())

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

EXPERIMENT_GROUPS = {
    "classical_baseline": [
        "configs/experiments/classical_largest_balanced_medleydb_mfcc_svm.yaml",
    ],
    "mert_head_ablation": [
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_linear.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4.yaml",
    ],
}

all_configs = sum(EXPERIMENT_GROUPS.values(), [])

print("Current working directory:", Path.cwd())
print("\nChecking config files:")

missing = []
for cfg_path in all_configs:
    path = Path(cfg_path)
    exists = path.is_file()
    print(cfg_path, "->", exists)
    if not exists:
        missing.append(cfg_path)

if missing:
    print("\nMissing configs:")
    for m in missing:
        print(" -", m)
    raise RuntimeError("Some config files are still missing. Check working directory/repository.")

print("\nExperiment IDs:")
for cfg_path in all_configs:
    with Path(cfg_path).open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    print(Path(cfg_path).name, "->", cfg["experiment_id"], "| approach:", cfg["approach"])

registry_path = RESULTS_DIR / "experiment_registry.csv"
print("\nRegistry exists:", registry_path.is_file())

if registry_path.is_file():
    registry = pd.read_csv(registry_path)
    existing_ids = set(registry["experiment_id"].astype(str))

    print("\nAlready registered among notebook 01 official configs:")
    for cfg_path in all_configs:
        with Path(cfg_path).open("r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
        print(cfg["experiment_id"], "->", cfg["experiment_id"] in existing_ids)

In [ ]:
from pathlib import Path
import subprocess

print("Current working directory:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", Path(PROJECT_ROOT).exists())

print("\nGit remote:")
subprocess.run(["git", "remote", "-v"], check=False)

print("\nCurrent branch:")
subprocess.run(["git", "branch", "--show-current"], check=False)

print("\nLast 5 commits:")
subprocess.run(["git", "log", "--oneline", "-5"], check=False)

print("\nRemote branches:")
subprocess.run(["git", "branch", "-r"], check=False)

print("\nconfigs/experiments content:")
configs_dir = Path("configs/experiments")
print("configs/experiments exists:", configs_dir.is_dir())

if configs_dir.is_dir():
    for p in sorted(configs_dir.glob("*.yaml")):
        print(p)
else:
    print("configs/experiments folder not found.")

print("\nAll configs:")
subprocess.run(
    "find configs -maxdepth 3 -type f | sort",
    shell=True,
    check=False,
)

In [ ]:
from pathlib import Path
import subprocess

commands = {
    "git status": ["git", "status"],
    "git remote -v": ["git", "remote", "-v"],
    "git branch -a": ["git", "branch", "-a"],
    "git log -5": ["git", "log", "--oneline", "-5"],
}

for name, cmd in commands.items():
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    print("RETURN CODE:", result.returncode)
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)

In [ ]:
from pathlib import Path

print("Notebooks available:")
for p in sorted(Path("notebooks").glob("*.ipynb")):
    print(p)

print("\nExperiment configs available:")
for p in sorted(Path("configs/experiments").glob("*.yaml")):
    print(p)

In [ ]:
!rm -rf /content/medleydb-mert-instrument-classification

In [ ]:
from pathlib import Path
import os
import sys
import shutil
import subprocess
from google.colab import userdata

# Paths già validati
PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

# Cancella SOLO la repo locale temporanea in /content, non il Drive
if PROJECT_ROOT.exists():
    print("Removing old local repository:", PROJECT_ROOT)
    shutil.rmtree(PROJECT_ROOT)

# Clona nuovamente la tua fork aggiornata
token = userdata.get("GITHUB_TOKEN")
if token is None:
    raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = "https://github.com/Frabbandera/medleydb-mert-instrument-classification.git"

print("Cloning updated repository...")
subprocess.run(["git", "clone", repo_url, str(PROJECT_ROOT)], check=True)

# Entra nella repo
os.chdir(PROJECT_ROOT)

# Installa requirements
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)

# Esporta path
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["HF_HOME"] = str(RUN_ROOT / "huggingface")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("\nSETUP COMPLETE")
print("Current working directory:", Path.cwd())
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())
print("RUN_ROOT exists:", RUN_ROOT.is_dir())
print("METADATA_DIR exists:", METADATA_DIR.is_dir())
print("CACHE_ROOT exists:", CACHE_ROOT.is_dir())
print("RESULTS_DIR exists:", RESULTS_DIR.is_dir())

In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")

if PROJECT_ROOT.exists():
    print("Removing old local repository:", PROJECT_ROOT)
    shutil.rmtree(PROJECT_ROOT)
else:
    print("Local repository not found, nothing to remove.")

print("PROJECT_ROOT exists after removal:", PROJECT_ROOT.exists())

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
from google.colab import userdata

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

token = userdata.get("GITHUB_TOKEN")
if token is None:
    raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = "https://github.com/Frabbandera/medleydb-mert-instrument-classification.git"

if PROJECT_ROOT.exists():
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
else:
    print("Repository not found. Cloning...")
    subprocess.run(["git", "clone", repo_url, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["HF_HOME"] = str(RUN_ROOT / "huggingface")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("\nSETUP COMPLETE")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("METADATA_DIR:", METADATA_DIR)
print("METADATA_DIR exists:", METADATA_DIR.is_dir())

print("CACHE_ROOT:", CACHE_ROOT)
print("CACHE_ROOT exists:", CACHE_ROOT.is_dir())

print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.is_dir())

print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")

if PROJECT_ROOT.exists():
    print("Removing incomplete repository:", PROJECT_ROOT)
    shutil.rmtree(PROJECT_ROOT)

print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

In [ ]:
from google.colab import userdata
import subprocess
import base64

token = userdata.get("GITHUB_TOKEN")
if token is None:
    raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets.")

repo_https = "https://github.com/Frabbandera/medleydb-mert-instrument-classification.git"

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
header = f"AUTHORIZATION: basic {auth}"

result = subprocess.run(
    [
        "git",
        "-c", f"http.https://github.com/.extraheader={header}",
        "ls-remote",
        repo_https,
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print("RETURN CODE:", result.returncode)
print("STDOUT preview:")
print(result.stdout[:1000])
print("STDERR:")
print(result.stderr[-2000:])

In [ ]:
from pathlib import Path
import os

os.chdir("/content")

print("Current working directory:", Path.cwd())
print("/content exists:", Path("/content").exists())
print("Old repo exists:", Path("/content/medleydb-mert-instrument-classification").exists())

In [ ]:
from google.colab import userdata
import subprocess
import base64
from pathlib import Path
import os

os.chdir("/content")

token = userdata.get("GITHUB_TOKEN")
if token is None:
    raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets.")

repo_https = "https://github.com/Frabbandera/medleydb-mert-instrument-classification.git"

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
header = f"AUTHORIZATION: basic {auth}"

result = subprocess.run(
    [
        "git",
        "-c", f"http.https://github.com/.extraheader={header}",
        "ls-remote",
        repo_https,
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print("RETURN CODE:", result.returncode)
print("STDOUT preview:")
print(result.stdout[:1000])
print("STDERR:")
print(result.stderr[-2000:])

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import base64
from google.colab import userdata

# Always start from a valid working directory
os.chdir("/content")

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

token = userdata.get("GITHUB_TOKEN")
if token is None:
    raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets.")

repo_https = "https://github.com/Frabbandera/medleydb-mert-instrument-classification.git"

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
header = f"AUTHORIZATION: basic {auth}"

if PROJECT_ROOT.exists():
    raise RuntimeError(f"PROJECT_ROOT already exists: {PROJECT_ROOT}. Remove it before cloning.")

result = subprocess.run(
    [
        "git",
        "-c", f"http.https://github.com/.extraheader={header}",
        "clone",
        repo_https,
        str(PROJECT_ROOT),
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print("CLONE RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout[-2000:])
print("STDERR:")
print(result.stderr[-2000:])

if result.returncode != 0:
    raise RuntimeError("Git clone failed. See STDERR above.")

os.chdir(PROJECT_ROOT)

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["HF_HOME"] = str(RUN_ROOT / "huggingface")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("\nSETUP COMPLETE")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("METADATA_DIR:", METADATA_DIR)
print("METADATA_DIR exists:", METADATA_DIR.is_dir())

print("CACHE_ROOT:", CACHE_ROOT)
print("CACHE_ROOT exists:", CACHE_ROOT.is_dir())

print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.is_dir())

print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path
import subprocess

print("Current working directory:", Path.cwd())

print("\nLatest commits:")
subprocess.run(["git", "log", "--oneline", "-5"], check=False)

target_configs = [
    "configs/experiments/classical_largest_balanced_medleydb_mfcc_svm.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_linear.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4.yaml",
]

print("\nTarget configs:")
for f in target_configs:
    print(f, "->", Path(f).is_file())

In [ ]:
required_pipeline_outputs = [
    METADATA_DIR / "subset_largest_balanced_medleydb_instrument.csv",
    METADATA_DIR / "labels_largest_balanced_medleydb_instrument_label_to_id.json",
    CACHE_ROOT / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "train.pt",
    CACHE_ROOT / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "val.pt",
    CACHE_ROOT / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "test.pt",
    RESULTS_DIR / "experiment_registry.csv",
]

for path in required_pipeline_outputs:
    print(path.name, "->", path.is_file())

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import os
import sys

# Make sure we are inside the updated repository
PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())

EXPERIMENT_GROUPS = {
    "classical_baseline": [
        "configs/experiments/classical_largest_balanced_medleydb_mfcc_svm.yaml",
    ],
    "mert_head_ablation": [
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_linear.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4.yaml",
    ],
}

all_configs = sum(EXPERIMENT_GROUPS.values(), [])

print("\nChecking config files:")
missing = []

for cfg_path in all_configs:
    path = Path(cfg_path)
    exists = path.is_file()
    print(cfg_path, "->", exists)
    if not exists:
        missing.append(cfg_path)

if missing:
    raise RuntimeError(f"Missing config files: {missing}")

print("\nExperiment IDs:")
for cfg_path in all_configs:
    with Path(cfg_path).open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    print(
        Path(cfg_path).name,
        "->",
        cfg["experiment_id"],
        "| approach:",
        cfg.get("approach"),
        "| task:",
        cfg.get("task"),
    )

registry_path = RESULTS_DIR / "experiment_registry.csv"
print("\nRegistry exists:", registry_path.is_file())

if registry_path.is_file():
    registry = pd.read_csv(registry_path)
    existing_ids = set(registry["experiment_id"].astype(str))

    print("\nAlready registered among notebook 01 official configs:")
    for cfg_path in all_configs:
        with Path(cfg_path).open("r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
        exp_id = cfg["experiment_id"]
        print(exp_id, "->", exp_id in existing_ids)

print("\nHelper functions:")
for name in ["run_selected", "run_experiment_config", "load_config"]:
    print(name, "->", name in globals())

In [ ]:
SELECTED_EXPERIMENTS = EXPERIMENT_GROUPS["classical_baseline"]
REPLACE_EXISTING = False

run_selected(
    SELECTED_EXPERIMENTS,
    extract_embeddings=False,
    replace_existing=REPLACE_EXISTING,
)

In [ ]:
from pathlib import Path
import json
import pandas as pd

classical_id = "classical_largest_balanced_medleydb_mfcc_svm"
classical_result_dir = RESULTS_DIR / classical_id
classical_checkpoint_dir = CHECKPOINTS_DIR / classical_id

print("Result dir exists:", classical_result_dir.is_dir())
print("Checkpoint dir exists:", classical_checkpoint_dir.is_dir())

for fname in [
    "test_metrics.json",
    "classification_report.csv",
    "confusion_matrix_raw.csv",
    "confusion_matrix_normalized.csv",
    "confusion_matrix_raw.png",
    "confusion_matrix_normalized.png",
    "predictions.csv",
    "config_resolved.yaml",
]:
    print(fname, "->", (classical_result_dir / fname).is_file())

if (classical_result_dir / "test_metrics.json").is_file():
    metrics = json.loads((classical_result_dir / "test_metrics.json").read_text())
    print("\nClassical baseline metrics:")
    print(json.dumps(metrics, indent=2))

registry = pd.read_csv(RESULTS_DIR / "experiment_registry.csv")
print("\nRegistry row:")
display(registry[registry["experiment_id"] == classical_id])

In [ ]:
SELECTED_EXPERIMENTS = [
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_linear.yaml"
]

REPLACE_EXISTING = False

run_selected(
    SELECTED_EXPERIMENTS,
    extract_embeddings=False,
    replace_existing=REPLACE_EXISTING,
)

In [ ]:
from pathlib import Path
import json
import pandas as pd

linear_id = "isolated_largest_balanced_medleydb_mert95_last_mean_linear"
linear_result_dir = RESULTS_DIR / linear_id
linear_checkpoint_dir = CHECKPOINTS_DIR / linear_id

print("Result dir exists:", linear_result_dir.is_dir())
print("Checkpoint dir exists:", linear_checkpoint_dir.is_dir())

for fname in [
    "test_metrics.json",
    "classification_report.csv",
    "confusion_matrix_raw.csv",
    "confusion_matrix_normalized.csv",
    "confusion_matrix_raw.png",
    "confusion_matrix_normalized.png",
    "predictions.csv",
    "config_resolved.yaml",
]:
    print(fname, "->", (linear_result_dir / fname).is_file())

if (linear_result_dir / "test_metrics.json").is_file():
    metrics = json.loads((linear_result_dir / "test_metrics.json").read_text())
    print("\nFrozen MERT linear metrics:")
    print(json.dumps(metrics, indent=2))

registry = pd.read_csv(RESULTS_DIR / "experiment_registry.csv")
print("\nRegistry row:")
display(registry[registry["experiment_id"] == linear_id])

In [ ]:
SELECTED_EXPERIMENTS = [
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4.yaml",
]

REPLACE_EXISTING = False

run_selected(
    SELECTED_EXPERIMENTS,
    extract_embeddings=False,
    replace_existing=REPLACE_EXISTING,
)

In [ ]:
from pathlib import Path
import json
import pandas as pd
import yaml

mlp_ids = [
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4",
]

for exp_id in mlp_ids:
    result_dir = RESULTS_DIR / exp_id
    checkpoint_dir = CHECKPOINTS_DIR / exp_id

    print("\n" + "=" * 100)
    print("Experiment:", exp_id)
    print("Result dir exists:", result_dir.is_dir())
    print("Checkpoint dir exists:", checkpoint_dir.is_dir())

    for fname in [
        "test_metrics.json",
        "classification_report.csv",
        "confusion_matrix_raw.csv",
        "confusion_matrix_normalized.csv",
        "confusion_matrix_raw.png",
        "confusion_matrix_normalized.png",
        "predictions.csv",
        "config_resolved.yaml",
    ]:
        print(fname, "->", (result_dir / fname).is_file())

    metrics_path = result_dir / "test_metrics.json"
    if metrics_path.is_file():
        metrics = json.loads(metrics_path.read_text())
        print("test_accuracy:", metrics.get("test_accuracy"))
        print("test_macro_f1:", metrics.get("test_macro_f1"))
        print("test_weighted_f1:", metrics.get("test_weighted_f1"))

In [ ]:
import pandas as pd
from pathlib import Path

notebook_01_ids = [
    "classical_largest_balanced_medleydb_mfcc_svm",
    "isolated_largest_balanced_medleydb_mert95_last_mean_linear",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4",
]

registry = pd.read_csv(RESULTS_DIR / "experiment_registry.csv")
comparison = registry[registry["experiment_id"].isin(notebook_01_ids)].copy()

comparison["test_macro_f1"] = pd.to_numeric(comparison["test_macro_f1"], errors="coerce")
comparison["test_accuracy"] = pd.to_numeric(comparison["test_accuracy"], errors="coerce")
comparison["test_weighted_f1"] = pd.to_numeric(comparison["test_weighted_f1"], errors="coerce")

comparison = comparison.sort_values("test_macro_f1", ascending=False)

cols = [
    "experiment_id",
    "approach",
    "model_name",
    "mert_layer",
    "pooling",
    "classifier_type",
    "hidden_dim",
    "dropout",
    "lr",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
]

display(comparison[[c for c in cols if c in comparison.columns]])

In [ ]:
EXPERIMENT_GROUPS = {
    "classical_baseline": [
        "configs/experiments/classical_largest_balanced_medleydb_mfcc_svm.yaml",
    ],
    "mert_head_ablation": [
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_linear.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4.yaml",
    ],
}
SELECTED_EXPERIMENTS = [
    "configs/experiments/classical_largest_balanced_medleydb_mfcc_svm.yaml",
]
REPLACE_EXISTING = False

run_selected(SELECTED_EXPERIMENTS, extract_embeddings=True, replace_existing=REPLACE_EXISTING)


## Registry Summary

This reads completed rows only; it is safe before any experiments have finished.

In [ ]:
# 1. Best-model error analysis

import pandas as pd
import json
from pathlib import Path
from IPython.display import Image, display
import matplotlib.pyplot as plt

notebook_01_ids = [
    "classical_largest_balanced_medleydb_mfcc_svm",
    "isolated_largest_balanced_medleydb_mert95_last_mean_linear",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4",
]

registry = pd.read_csv(RESULTS_DIR / "experiment_registry.csv")
comparison = registry[registry["experiment_id"].isin(notebook_01_ids)].copy()

for col in ["test_accuracy", "test_macro_f1", "test_weighted_f1"]:
    comparison[col] = pd.to_numeric(comparison[col], errors="coerce")

comparison = comparison.sort_values("test_macro_f1", ascending=False)

def short_name(exp_id):
    if exp_id == "classical_largest_balanced_medleydb_mfcc_svm":
        return "MFCC+SVM"
    if exp_id.endswith("_linear"):
        return "MERT linear"
    if exp_id.endswith("_mlp_h256_d00"):
        return "MERT MLP h256 d0.0"
    if exp_id.endswith("_mlp_h256_d02"):
        return "MERT MLP h256 d0.2"
    if exp_id.endswith("_mlp_h256_d05"):
        return "MERT MLP h256 d0.5"
    if exp_id.endswith("_mlp_h512_d02"):
        return "MERT MLP h512 d0.2"
    if exp_id.endswith("_mlp_h256_d02_lr3e4"):
        return "MERT MLP h256 d0.2 lr3e-4"
    return exp_id

comparison["short_name"] = comparison["experiment_id"].apply(short_name)

display_cols = [
    "short_name",
    "experiment_id",
    "approach",
    "classifier_type",
    "hidden_dim",
    "dropout",
    "lr",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
]

display(comparison[[c for c in display_cols if c in comparison.columns]])

best_overall = comparison.iloc[0]
best_overall_id = best_overall["experiment_id"]

mert_comparison = comparison[comparison["approach"] == "frozen_embeddings"].copy()
best_mert = mert_comparison.iloc[0]
best_mert_id = best_mert["experiment_id"]

print("Best overall model:", short_name(best_overall_id), "|", best_overall_id)
print("Best frozen MERT model:", short_name(best_mert_id), "|", best_mert_id)

In [ ]:
# 2. Best-overall-model error analysis

best_id = best_overall_id
best_dir = RESULTS_DIR / best_id

print("Analyzing best overall model:")
print(best_id)

metrics = json.loads((best_dir / "test_metrics.json").read_text())
print(json.dumps(metrics, indent=2))

report = pd.read_csv(best_dir / "classification_report.csv", index_col=0)
display(report)

raw = pd.read_csv(best_dir / "confusion_matrix_raw.csv", index_col=0)
normalized = pd.read_csv(best_dir / "confusion_matrix_normalized.csv", index_col=0)

print("Raw confusion matrix:")
display(raw)

print("Normalized confusion matrix:")
display(normalized)

display(Image(filename=str(best_dir / "confusion_matrix_raw.png")))
display(Image(filename=str(best_dir / "confusion_matrix_normalized.png")))

class_rows = [row for row in report.index if row not in {"accuracy", "macro avg", "weighted avg"}]

ax = report.loc[class_rows, "f1-score"].sort_values().plot.barh(
    figsize=(8, 5),
    title=f"Per-class F1: {short_name(best_id)}"
)
ax.set_xlabel("F1 score")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

print("Worst classes by F1:")
display(report.loc[class_rows, ["precision", "recall", "f1-score", "support"]].sort_values("f1-score").head(12))

In [ ]:
# 3. Best-overall-model most frequent mistakes

predictions = pd.read_csv(best_dir / "predictions.csv")

print("Predictions shape:", predictions.shape)
display(predictions.head())

if "correct" in predictions.columns:
    errors = predictions[predictions["correct"] == False].copy()
else:
    errors = predictions[predictions["true_label"] != predictions["predicted_label"]].copy()

print("Number of errors:", len(errors))

error_pairs = (
    errors.groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(error_pairs.head(20))

error_pairs.to_csv(RESULTS_DIR / "notebook_01_best_overall_error_pairs.csv", index=False)
comparison.to_csv(RESULTS_DIR / "notebook_01_classifier_head_comparison.csv", index=False)

print("Saved:")
print(RESULTS_DIR / "notebook_01_best_overall_error_pairs.csv")
print(RESULTS_DIR / "notebook_01_classifier_head_comparison.csv")

In [ ]:
# 4. Best-frozen-MERT-model error analysis

best_mert_dir = RESULTS_DIR / best_mert_id

print("Analyzing best frozen MERT model:")
print(best_mert_id)
print(short_name(best_mert_id))

mert_metrics = json.loads((best_mert_dir / "test_metrics.json").read_text())
print(json.dumps(mert_metrics, indent=2))

mert_report = pd.read_csv(best_mert_dir / "classification_report.csv", index_col=0)
display(mert_report)

mert_raw = pd.read_csv(best_mert_dir / "confusion_matrix_raw.csv", index_col=0)
mert_normalized = pd.read_csv(best_mert_dir / "confusion_matrix_normalized.csv", index_col=0)

display(Image(filename=str(best_mert_dir / "confusion_matrix_raw.png")))
display(Image(filename=str(best_mert_dir / "confusion_matrix_normalized.png")))

class_rows = [row for row in mert_report.index if row not in {"accuracy", "macro avg", "weighted avg"}]

ax = mert_report.loc[class_rows, "f1-score"].sort_values().plot.barh(
    figsize=(8, 5),
    title=f"Per-class F1: {short_name(best_mert_id)}"
)
ax.set_xlabel("F1 score")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

print("Worst MERT classes by F1:")
display(mert_report.loc[class_rows, ["precision", "recall", "f1-score", "support"]].sort_values("f1-score").head(12))

mert_predictions = pd.read_csv(best_mert_dir / "predictions.csv")

if "correct" in mert_predictions.columns:
    mert_errors = mert_predictions[mert_predictions["correct"] == False].copy()
else:
    mert_errors = mert_predictions[mert_predictions["true_label"] != mert_predictions["predicted_label"]].copy()

mert_error_pairs = (
    mert_errors.groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(mert_error_pairs.head(20))

mert_error_pairs.to_csv(RESULTS_DIR / "notebook_01_best_mert_error_pairs.csv", index=False)

print("Saved:")
print(RESULTS_DIR / "notebook_01_best_mert_error_pairs.csv")

In [ ]:
# 5. Final sanity-check

from pathlib import Path
import pandas as pd

notebook_01_ids = [
    "classical_largest_balanced_medleydb_mfcc_svm",
    "isolated_largest_balanced_medleydb_mert95_last_mean_linear",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d00",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d05",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h512_d02",
    "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_lr3e4",
]

required_result_files = [
    "test_metrics.json",
    "classification_report.csv",
    "confusion_matrix_raw.csv",
    "confusion_matrix_normalized.csv",
    "confusion_matrix_raw.png",
    "confusion_matrix_normalized.png",
    "predictions.csv",
    "config_resolved.yaml",
]

all_ok = True

for exp_id in notebook_01_ids:
    result_dir = RESULTS_DIR / exp_id
    checkpoint_dir = CHECKPOINTS_DIR / exp_id

    print("\n" + "=" * 100)
    print("Experiment:", exp_id)
    print("Result dir exists:", result_dir.is_dir())
    print("Checkpoint dir exists:", checkpoint_dir.is_dir())

    if not result_dir.is_dir():
        all_ok = False

    for fname in required_result_files:
        exists = (result_dir / fname).is_file()
        print(f"{fname:35s} -> {exists}")
        if not exists:
            all_ok = False

print("\n" + "=" * 100)
print("Notebook 01 summary files")
print("=" * 100)

summary_files = [
    RESULTS_DIR / "notebook_01_classifier_head_comparison.csv",
    RESULTS_DIR / "notebook_01_best_overall_error_pairs.csv",
    RESULTS_DIR / "notebook_01_best_mert_error_pairs.csv",
]

for path in summary_files:
    exists = path.is_file()
    print(path.name, "->", exists)
    if not exists:
        all_ok = False

registry_path = RESULTS_DIR / "experiment_registry.csv"
print("\nexperiment_registry.csv ->", registry_path.is_file())

if registry_path.is_file():
    registry = pd.read_csv(registry_path)
    registered = set(registry["experiment_id"].astype(str))

    print("\nRegistered notebook 01 experiments:")
    for exp_id in notebook_01_ids:
        print(exp_id, "->", exp_id in registered)
        if exp_id not in registered:
            all_ok = False

print("\nFINAL NOTEBOOK 01 STATUS:", "OK" if all_ok else "CHECK NEEDED")

In [ ]:
from pathlib import Path

RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

important_paths = [
    RUN_ROOT / "data" / "metadata" / "stem_index.csv",
    RUN_ROOT / "data" / "metadata" / "subset_largest_balanced_medleydb_instrument.csv",
    RUN_ROOT / "data" / "metadata" / "labels_largest_balanced_medleydb_instrument_label_to_id.json",
    RUN_ROOT / "data" / "cache" / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "train.pt",
    RUN_ROOT / "data" / "cache" / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "val.pt",
    RUN_ROOT / "data" / "cache" / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "test.pt",
    RUN_ROOT / "results" / "experiment_registry.csv",
    RUN_ROOT / "results" / "notebook_01_classifier_head_comparison.csv",
    RUN_ROOT / "results" / "notebook_01_best_overall_error_pairs.csv",
    RUN_ROOT / "results" / "notebook_01_best_mert_error_pairs.csv",
]

for path in important_paths:
    print(path.relative_to(RUN_ROOT), "->", path.is_file())

print("\nTotal files in RUN_ROOT:", sum(1 for p in RUN_ROOT.rglob("*") if p.is_file()))

In [ ]:
registry_path = RESULTS_DIR / "experiment_registry.csv"
if registry_path.exists():
    registry = pd.read_csv(registry_path)
    display(registry[registry["experiment_id"].isin([Path(p).stem for p in sum(EXPERIMENT_GROUPS.values(), [])])].tail(20))
else:
    print("No registry yet:", registry_path)
